# Programa C — Preparación de datos y entrenamiento

## Propósito

En esta actividad completaremos el flujo desde las imágenes etiquetadas hasta un modelo de clasificación.

La secuencia será:

```text
imágenes + etiquetas
        ↓
carga en Python
        ↓
representación numérica
        ↓
X e y
        ↓
normalización
        ↓
entrenamiento / prueba
        ↓
red neuronal
        ↓
predicción
        ↓
evaluación
```

> Este ejercicio utiliza solamente 60 imágenes. Su objetivo es **comprender el proceso**, no obtener un modelo de producción.


## 1. Importar librerías


In [ ]:
# Si fuera necesario instalar dependencias:
# %pip install pandas numpy pillow matplotlib scikit-learn

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay


## 2. Definir rutas y cargar las etiquetas


In [ ]:
BASE = Path("dataset_etiquetado")
CARPETA_IMAGENES = BASE / "imagenes"
ARCHIVO_ETIQUETAS = CARPETA_IMAGENES / "etiquetas.csv"

df = pd.read_csv(ARCHIVO_ETIQUETAS)

display(df.head())
print("Cantidad de registros:", len(df))


## 3. Cargar una imagen y observar qué contiene

Una imagen digital puede verse como una matriz de números.

Cada número representa la intensidad de un píxel.


In [ ]:
primera_ruta = CARPETA_IMAGENES / df.iloc[0]["archivo"]

imagen = Image.open(primera_ruta).convert("L")
matriz = np.array(imagen)

print("Archivo:", primera_ruta.name)
print("Forma de la imagen:", matriz.shape)
print("Tipo de dato:", matriz.dtype)
print("Valor mínimo:", matriz.min())
print("Valor máximo:", matriz.max())

plt.imshow(matriz, cmap="gray")
plt.title(f'Etiqueta: {df.iloc[0]["etiqueta"]}')
plt.axis("off")
plt.show()


## 4. Transformar todas las imágenes en datos numéricos

Fashion-MNIST utiliza imágenes de **28 × 28 píxeles**.

Cada imagen contiene:

\[
28 \times 28 = 784
\]

valores numéricos.

Para utilizar una red neuronal MLP convertiremos cada matriz de 28 × 28 en una fila de 784 características.


In [ ]:
X_lista = []
y_lista = []

for _, fila in df.iterrows():
    ruta = CARPETA_IMAGENES / fila["archivo"]

    imagen = Image.open(ruta).convert("L")
    matriz = np.array(imagen)

    # Convertir 28x28 en un vector de 784 valores
    vector = matriz.reshape(-1)

    X_lista.append(vector)
    y_lista.append(fila["etiqueta"])

X = np.array(X_lista)
y = np.array(y_lista)

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)
print("Ejemplo de etiqueta:", y[0])


### ¿Qué significa `X.shape = (60, 784)`?

- **60** corresponde a la cantidad de imágenes.
- **784** corresponde a los píxeles de cada imagen después de aplanarla.

La etiqueta correspondiente a cada fila de `X` se encuentra en la misma posición de `y`.


## 5. Normalizar los valores de los píxeles

Los píxeles originales poseen valores entre 0 y 255.

Dividiremos por 255 para transformarlos aproximadamente al rango 0–1.


In [ ]:
print("Antes de normalizar:")
print("Mínimo:", X.min())
print("Máximo:", X.max())

X = X.astype("float32") / 255.0

print("\nDespués de normalizar:")
print("Mínimo:", X.min())
print("Máximo:", X.max())


## 6. Separar entrenamiento y prueba

Reservaremos el 20 % de las imágenes para probar el modelo.

Usaremos `stratify=y` para intentar mantener la proporción de clases en ambos grupos.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


## 7. Revisar la distribución de clases después de separar


In [ ]:
print("Entrenamiento:")
display(pd.Series(y_train).value_counts().sort_index())

print("Prueba:")
display(pd.Series(y_test).value_counts().sort_index())


## 8. Crear una red neuronal

Utilizaremos `MLPClassifier`, al igual que en la práctica anterior con Digits.

En este caso la capa de entrada recibe 784 características por imagen.

La capa oculta será una decisión de diseño del modelo.


In [ ]:
modelo = MLPClassifier(
    hidden_layer_sizes=(32,),
    max_iter=1000,
    random_state=42
)

modelo


## 9. Entrenar

Durante `fit()` la red recibe:

- `X_train`: las imágenes convertidas a números;
- `y_train`: las etiquetas correspondientes.

A partir de ambos conjuntos ajusta internamente sus pesos.


In [ ]:
modelo.fit(X_train, y_train)

print("Entrenamiento finalizado.")
print("Iteraciones realizadas:", modelo.n_iter_)


## 10. Generar predicciones

Ahora entregaremos al modelo imágenes que no utilizó durante el entrenamiento.


In [ ]:
predicciones = modelo.predict(X_test)

resultado = pd.DataFrame({
    "real": y_test,
    "predicha": predicciones
})

display(resultado)


## 11. Calcular exactitud


In [ ]:
exactitud = accuracy_score(y_test, predicciones)

print(f"Exactitud: {exactitud:.4f}")
print(f"Exactitud porcentual: {exactitud * 100:.2f}%")


## 12. Matriz de confusión

La matriz de confusión permite observar qué clases fueron reconocidas correctamente y cuáles se confundieron.


In [ ]:
clases = sorted(df["etiqueta"].unique())

matriz_confusion = confusion_matrix(
    y_test,
    predicciones,
    labels=clases
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz_confusion,
    display_labels=clases
)

disp.plot()
plt.title("Matriz de confusión")
plt.xlabel("Etiqueta predicha")
plt.ylabel("Etiqueta real")
plt.show()


## 13. Mostrar imágenes de prueba con su resultado


In [ ]:
columnas = 4
filas = (len(X_test) + columnas - 1) // columnas

plt.figure(figsize=(12, 3 * filas))

for i in range(len(X_test)):
    plt.subplot(filas, columnas, i + 1)

    imagen = X_test[i].reshape(28, 28)

    plt.imshow(imagen, cmap="gray")
    plt.title(
        f"Real: {y_test[i]}\n"
        f"Predijo: {predicciones[i]}"
    )
    plt.axis("off")

plt.tight_layout()
plt.show()


## 14. Analizar únicamente los errores


In [ ]:
indices_error = np.where(predicciones != y_test)[0]

print("Cantidad de errores:", len(indices_error))

if len(indices_error) == 0:
    print("El modelo clasificó correctamente todas las imágenes de prueba.")
else:
    columnas = 4
    filas = (len(indices_error) + columnas - 1) // columnas

    plt.figure(figsize=(12, 3 * filas))

    for posicion, indice in enumerate(indices_error, start=1):
        plt.subplot(filas, columnas, posicion)

        imagen = X_test[indice].reshape(28, 28)

        plt.imshow(imagen, cmap="gray")
        plt.title(
            f"Real: {y_test[indice]}\n"
            f"Predijo: {predicciones[indice]}"
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()


## 15. Preguntas de análisis

1. ¿Qué representa cada fila de `X`?
2. ¿Por qué cada imagen genera 784 características?
3. ¿Qué relación existe entre `X[i]` e `y[i]`?
4. ¿Para qué normalizamos los valores de los píxeles?
5. ¿Por qué separamos entrenamiento y prueba?
6. ¿Qué significa que el modelo se equivoque en una imagen de prueba?
7. ¿El resultado obtenido permitiría afirmar que el modelo generalizará correctamente a miles de imágenes nuevas? ¿Por qué?
8. ¿Qué podría ocurrir si algunas etiquetas utilizadas para entrenar estuvieran equivocadas?


## Idea clave

En la práctica anterior con `load_digits()`, `X` e `y` ya venían preparados.

En esta actividad construimos nosotros mismos esa relación:

```text
archivo PNG
    ↓
matriz 28 × 28
    ↓
vector de 784 valores
    ↓
X

etiqueta manual
    ↓
y
```

A partir de allí el proceso vuelve a ser el conocido:

**preparar → separar → entrenar → predecir → evaluar**.
